# Construction Safety — API smoke tests (`call_api.ipynb`)

Test **IMU** and **keypoint** inference against a locally running API or your deployed Render service, using **real VTT-ConIoT CSV data**. The notebook checks endpoint availability, request/response shape, and (optionally) local-vs-Render agreement.

**Source alignment:** Your uploaded `model_app/app.py` is currently a **Gradio** IMU-only server exposing `predict` and `embed` through Gradio's API. The proposed `02_model_server/` FastAPI server is expected to expose `GET /health`, `POST /predict/imu`, and `POST /predict/keypoints`. This notebook supports both modes, but **does not imply that the new server or keypoint endpoint already exists**.

**Interpretation:** Reconstruction error measures signal reconstruction, **not diagnosed fatigue, psychological stress, or injury**. Do not submit personal recordings to a public endpoint without appropriate authorization.

## 1. Install dependencies

Run once in your selected VS Code/Jupyter environment (uncomment as needed).

In [5]:
# %pip install requests pandas numpy gradio_client

## 2. Configure target and locate repository files

- Set `API_MODE = "fastapi"` for the planned `02_model_server/` API.
- Set `API_MODE = "gradio"` for the **existing** `model_app/app.py` (IMU only).
- `API_URL` should be `http://127.0.0.1:8000` for local FastAPI or your real `https://...onrender.com` URL.
- Do **not** paste Hugging Face tokens here. The server owns model access.

In [6]:
%pip install requests pandas numpy gradio_client ipykernel

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
from pathlib import Path
import json, os, re, time
import numpy as np
import pandas as pd
import requests
from IPython.display import display

API_MODE = "fastapi"       # "fastapi" or "gradio"
API_URL = os.getenv("SAFETY_API_URL", "http://127.0.0.1:8000").rstrip("/")
REQUEST_TIMEOUT = 120       # Render free services may need time to wake up

# If the notebook is saved in 02_model_server/, repo root is its parent.
HERE = Path.cwd().resolve()
ROOT = next((p for p in [HERE, *HERE.parents] if (p / "01_autoencoder").exists() or (p / "model_app").exists()), HERE)
print("Repository root:", ROOT)
print("Target:", API_MODE, API_URL)


Repository root: C:\Users\Akanksha\Documents\IAAC\CLASSES\AI Methods\Hackathon\Hackaton_Group3_AIMethods
Target: fastapi http://127.0.0.1:8000


## 3. Check server connectivity

A `404` for `/health` in Gradio mode is not necessarily an error; the existing Gradio app has no `/health` endpoint.

In [8]:
def get_json(path, base=API_URL):
    response = requests.get(base.rstrip("/") + path, timeout=REQUEST_TIMEOUT)
    print("GET", path, "→", response.status_code)
    response.raise_for_status()
    try:
        return response.json()
    except ValueError:
        return response.text[:500]

try:
    health = get_json("/health" if API_MODE == "fastapi" else "/")
    print(str(health)[:1000])
except requests.RequestException as exc:
    print("Connection/health check failed:", exc)
    print("Start the server locally or set SAFETY_API_URL to the deployed Render URL.")


Connection/health check failed: HTTPConnectionPool(host='127.0.0.1', port=8000): Max retries exceeded with url: /health (Caused by NewConnectionError("HTTPConnection(host='127.0.0.1', port=8000): Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it"))
Start the server locally or set SAFETY_API_URL to the deployed Render URL.


## 4. Load a **real IMU window** (200 × 18)

The order comes directly from your uploaded `model_app/config.json`. We use the **raw unscaled sensor values**: the existing IMU server applies its saved `scaler.npz` internally. The new server must do the same.

In [9]:
CONFIG_PATH = ROOT / "model_app" / "config.json"
if CONFIG_PATH.exists():
    imu_config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
    FEATURES = imu_config["features"]
    WINDOW = int(imu_config["window"])
else:
    FEATURES = [f"{loc}_{channel}" for loc in ("trousers", "back", "hand")
                for channel in ("Ax_g", "Ay_g", "Az_g", "Gx_dps", "Gy_dps", "Gz_dps")]
    WINDOW = 200
print("IMU shape expected:", (WINDOW, len(FEATURES)))
print("Channel order:", FEATURES)

# Change this to a specific real IMU CSV if automatic discovery selects the wrong file.
IMU_CSV = None

def find_imu_csv():
    roots = [ROOT / "data", ROOT / "File conversions" / "VTT_ConIot_Dataset"]
    for directory in roots:
        if not directory.exists():
            continue
        for path in directory.rglob("*.csv"):
            try:
                head = pd.read_csv(path, nrows=1)
                if set(FEATURES).issubset(head.columns):
                    return path
            except (OSError, ValueError, pd.errors.ParserError):
                continue
    return None

imu_path = Path(IMU_CSV) if IMU_CSV else find_imu_csv()
imu_window = None
if imu_path and imu_path.exists():
    frame = pd.read_csv(imu_path)
    if not set(FEATURES).issubset(frame.columns):
        raise ValueError("Selected IMU CSV does not contain all required features")
    values = frame[FEATURES].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float32)
    if len(values) < WINDOW:
        raise ValueError(f"Need at least {WINDOW} IMU rows")
    imu_window = values[:WINDOW]
    if not np.isfinite(imu_window).all():
        raise ValueError("Selected IMU window contains NaN/inf")
    print("Loaded:", imu_path, "shape:", imu_window.shape)
else:
    print("No local IMU CSV found. Set IMU_CSV to a real dataset CSV before testing inference.")


IMU shape expected: (200, 18)
Channel order: ['trousers_Ax_g', 'trousers_Ay_g', 'trousers_Az_g', 'trousers_Gx_dps', 'trousers_Gy_dps', 'trousers_Gz_dps', 'back_Ax_g', 'back_Ay_g', 'back_Az_g', 'back_Gx_dps', 'back_Gy_dps', 'back_Gz_dps', 'hand_Ax_g', 'hand_Ay_g', 'hand_Az_g', 'hand_Gx_dps', 'hand_Gy_dps', 'hand_Gz_dps']
Loaded: C:\Users\Akanksha\Documents\IAAC\CLASSES\AI Methods\Hackathon\Hackaton_Group3_AIMethods\data\VTT_ConIot_Dataset\VTT_ConIot_Dataset\VTT-ConIot-IMU-Data\activity_10_user_10_combined.csv shape: (200, 18)


## 5. Call the IMU endpoint

The proposed FastAPI request contract is `{"window": [[...], ...]}`. **Change `IMU_PAYLOAD_KEY` to `windows` if your team's final server implements a batched contract**. The existing Gradio API is called through `gradio_client` using its published `predict` endpoint.

In [10]:
IMU_PAYLOAD_KEY = "window"  # expected single-window FastAPI contract; adjust to actual app.py

def call_imu(window, base=API_URL, mode=API_MODE):
    if window is None:
        raise ValueError("No real IMU window loaded")
    if mode == "gradio":
        from gradio_client import Client
        client = Client(base)
        return client.predict(window.tolist(), api_name="/predict")
    payload = {IMU_PAYLOAD_KEY: window.tolist()}
    response = requests.post(base.rstrip("/") + "/predict/imu", json=payload, timeout=REQUEST_TIMEOUT)
    print("IMU HTTP status:", response.status_code)
    if not response.ok:
        print("Response:", response.text[:1500])
    response.raise_for_status()
    return response.json()

imu_result = None
if imu_window is not None:
    try:
        imu_result = call_imu(imu_window)
        print(json.dumps(imu_result, indent=2)[:1800])
    except Exception as exc:
        print("IMU call failed:", type(exc).__name__, str(exc)[:800])
        print("Check API_MODE, URL, endpoint and request key against the running server.")
else:
    print("Skipped: real IMU CSV not available")


IMU call failed: ConnectionError HTTPConnectionPool(host='127.0.0.1', port=8000): Max retries exceeded with url: /predict/imu (Caused by NewConnectionError("HTTPConnection(host='127.0.0.1', port=8000): Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it"))
Check API_MODE, URL, endpoint and request key against the running server.


## 6. Validate IMU response

The existing Gradio `predict` returns `reconstruction_mse_standardized` and `reconstruction_raw`. The final FastAPI server may use a different schema: adjust checks after finalizing the API contract.

In [11]:
if isinstance(imu_result, dict):
    if "reconstruction_mse_standardized" in imu_result:
        mse = float(imu_result["reconstruction_mse_standardized"])
        assert np.isfinite(mse) and mse >= 0
        print("IMU standardized reconstruction MSE:", mse)
    if "reconstruction_raw" in imu_result:
        reconstruction = np.asarray(imu_result["reconstruction_raw"], dtype=float)
        assert reconstruction.shape == (WINDOW, len(FEATURES)), reconstruction.shape
        assert np.isfinite(reconstruction).all()
        print("IMU reconstruction shape verified:", reconstruction.shape)
    print("IMU response keys:", sorted(imu_result))
else:
    print("No IMU response available to validate")


No IMU response available to validate


## 7. Load **real keypoint frames** (17 COCO joints × x/y/confidence)

Your uploaded `keypoint_utils.py` expects `x0,y0,prob0,...,x16,y16,prob16` in the source CSV. This notebook sends raw **48 frames × 17 joints × 3 values** to the proposed API; the server should apply the project's `clean_clip` and `preprocess_for_model` logic, using the **keypoint scaler from training**. If the API accepts preprocessed 48 × 34 windows instead, modify the payload contract accordingly.

In [12]:
KEYPOINT_CSV = None  # optional: absolute or repo-relative path to a real keypoint CSV
KEYPOINT_FRAMES = 48

def find_keypoint_csv():
    required = {f"{axis}{i}" for i in range(17) for axis in ("x", "y", "prob")}
    for directory in [ROOT / "data", ROOT / "File conversions" / "VTT_ConIot_Dataset"]:
        if not directory.exists():
            continue
        for path in directory.rglob("*.csv"):
            try:
                if required.issubset(pd.read_csv(path, nrows=1).columns):
                    return path
            except (OSError, ValueError, pd.errors.ParserError):
                continue
    return None

kp_path = Path(KEYPOINT_CSV) if KEYPOINT_CSV else find_keypoint_csv()
keypoint_frames = None
if kp_path and kp_path.exists():
    kp_df = pd.read_csv(kp_path)
    if len(kp_df) < KEYPOINT_FRAMES:
        raise ValueError("Keypoint CSV is shorter than 48 frames")
    xyz = np.stack([kp_df[[f"{axis}{i}" for i in range(17)]].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float32)
                    for axis in ("x", "y", "prob")], axis=-1)
    keypoint_frames = xyz[:KEYPOINT_FRAMES]
    if not np.isfinite(keypoint_frames).all():
        raise ValueError("Keypoint input contains NaN/inf")
    print("Loaded:", kp_path, "shape:", keypoint_frames.shape)
else:
    print("No keypoint CSV found; set KEYPOINT_CSV before testing keypoint inference")


Loaded: C:\Users\Akanksha\Documents\IAAC\CLASSES\AI Methods\Hackathon\Hackaton_Group3_AIMethods\data\VTT_ConIot_Dataset\VTT_ConIot_Dataset\VTT-ConIoT-Video-Keypoints\Subject_01_Task_1.m2ts_keyPoints.csv shape: (48, 17, 3)


## 8. Call the keypoint endpoint (planned FastAPI server only)

The current `model_app/app.py` **does not implement keypoint inference**. A `404` or `503` here means the new server is not yet implemented or its model artifacts are unavailable. This is not a successful model test.

In [13]:
KEYPOINT_PAYLOAD_KEY = "frames"  # planned raw-frame contract

def call_keypoints(frames, base=API_URL):
    if frames is None:
        raise ValueError("No real keypoint frames loaded")
    response = requests.post(
        base.rstrip("/") + "/predict/keypoints",
        json={KEYPOINT_PAYLOAD_KEY: frames.tolist()}, timeout=REQUEST_TIMEOUT
    )
    print("Keypoint HTTP status:", response.status_code)
    if not response.ok:
        print("Response:", response.text[:1500])
    response.raise_for_status()
    return response.json()

kp_result = None
if API_MODE != "fastapi":
    print("Skipped: existing Gradio server has no keypoint endpoint")
elif keypoint_frames is None:
    print("Skipped: no real keypoint sample")
else:
    try:
        kp_result = call_keypoints(keypoint_frames)
        print(json.dumps(kp_result, indent=2)[:1800])
    except Exception as exc:
        print("Keypoint call failed:", type(exc).__name__, str(exc)[:800])
        print("Check whether the new keypoint endpoint and Hugging Face artifacts are deployed.")


Keypoint call failed: ConnectionError HTTPConnectionPool(host='127.0.0.1', port=8000): Max retries exceeded with url: /predict/keypoints (Caused by NewConnectionError("HTTPConnection(host='127.0.0.1', port=8000): Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it"))
Check whether the new keypoint endpoint and Hugging Face artifacts are deployed.


## 9. Check keypoint response

The precise output schema depends on your final server. This cell verifies a JSON object and reports returned fields without inventing a fatigue/stress classification.

In [14]:
if kp_result is not None:
    assert isinstance(kp_result, dict), "Expected JSON object from keypoint endpoint"
    print("Keypoint response keys:", sorted(kp_result))
    for name in ("reconstruction_mse_standardized", "anomaly_error_median", "threshold"):
        if name in kp_result:
            value = float(kp_result[name])
            assert np.isfinite(value)
            print(f"{name}: {value}")
else:
    print("No keypoint prediction available to validate")


No keypoint prediction available to validate


## 10. Optional: compare localhost with Render

Run **only after both servers use the same model version and preprocessing**. This compares the scalar reconstruction error if both responses expose the same field; small floating-point differences can occur.

In [15]:
RENDER_URL = os.getenv("SAFETY_RENDER_URL", "").rstrip("/")  # e.g. https://your-service.onrender.com
LOCAL_URL = "http://127.0.0.1:8000"

if RENDER_URL and imu_window is not None and API_MODE == "fastapi":
    try:
        local = call_imu(imu_window, base=LOCAL_URL)
        remote = call_imu(imu_window, base=RENDER_URL)
        field = "reconstruction_mse_standardized"
        if field in local and field in remote:
            delta = abs(float(local[field]) - float(remote[field]))
            print("Absolute local/Render IMU MSE difference:", delta)
            assert np.isclose(local[field], remote[field], rtol=1e-4, atol=1e-5)
        else:
            print("Comparison skipped: both responses must contain", field)
    except Exception as exc:
        print("Local/Render comparison failed:", type(exc).__name__, str(exc)[:800])
else:
    print("Optional comparison skipped. Set SAFETY_RENDER_URL and start the local FastAPI server.")


Optional comparison skipped. Set SAFETY_RENDER_URL and start the local FastAPI server.


## 11. Deployment checklist

- [ ] `02_model_server/app.py` serves `GET /health`, `POST /predict/imu`, `POST /predict/keypoints`.
- [ ] IMU preprocessing matches `model_app/config.json` and `scaler.npz` (200 × 18).
- [ ] Keypoint preprocessing matches `01_autoencoder/keypoint_utils.py` (48 frames, 17 joints, 34 normalized features).
- [ ] Hugging Face model files and access credentials are configured **server-side**.
- [ ] Real IMU and keypoint samples both return valid predictions locally.
- [ ] Render Blueprint is live; hosted responses agree with local results.
- [ ] API responses describe **movement/reconstruction indicators**, not medically validated fatigue or stress diagnoses.

**Git:** save this notebook as `02_model_server/call_api.ipynb`; commit the notebook only, not the private dataset or secrets.